# Prompt Technique Effects on Model Behavior and Performance

This notebook analyzes how prompt-template techniques affect model outputs in `results/<model>_final.jsonl`.

It joins:

- `problems.csv`: problem/task metadata and tests
- `prompt_matrix_prompt_template.08.csv`: prompt-template factors (`Tone`, `Role framing`, `Format`, and any non-constant factor columns)
- `results/*_final.jsonl`: model outputs and evaluation results

The main questions are:

1. Which prompt techniques improve correctness and test pass fraction?
2. Which techniques change behavior, such as verbosity, code extraction, runtime, token use, and error modes?
3. Are effects consistent across models, or model-specific?

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import statsmodels.formula.api as smf
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
PROBLEMS_PATH = ROOT / "problems.csv"
PROMPT_MATRIX_PATH = ROOT / "prompt_matrix_prompt_template.08.csv"
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = ROOT / "analysis_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42

## Load and Join Data

Result `task_id` values are expected to look like `problemID_templateID`, for example `1_7`. The notebook parses those IDs and joins template factors by `template_id == prompt_matrix.ID`.

In [ ]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} line {line_no}: {exc}") from exc
            row["source_file"] = path.name
            rows.append(row)
    return pd.DataFrame(rows)


def short_model_name(source_file: str, model_value: str | None = None) -> str:
    if source_file.endswith("_final.jsonl"):
        return source_file.removesuffix("_final.jsonl")
    if model_value:
        return str(model_value).replace(":", "-")
    return source_file.removesuffix(".jsonl")


def parse_result_task_id(value: object) -> tuple[int, int]:
    text = str(value)
    match = re.fullmatch(r"(\d+)_(\d+)", text)
    if not match:
        raise ValueError(f"Unexpected result task_id format: {text!r}")
    return int(match.group(1)), int(match.group(2))

problems = pd.read_csv(PROBLEMS_PATH)
prompt_matrix = pd.read_csv(PROMPT_MATRIX_PATH)
result_paths = sorted(RESULTS_DIR.glob("*_final.jsonl"))

if not result_paths:
    raise FileNotFoundError(f"No *_final.jsonl files found in {RESULTS_DIR}")

results = pd.concat([read_jsonl(path) for path in result_paths], ignore_index=True)

parsed_ids = results["task_id"].map(parse_result_task_id)
results["problem_id"] = parsed_ids.map(lambda pair: pair[0])
results["template_id"] = parsed_ids.map(lambda pair: pair[1])
results["model_short"] = [short_model_name(src, model) for src, model in zip(results["source_file"], results.get("model", pd.Series([None] * len(results))))]

prompt_matrix = prompt_matrix.rename(columns={
    "ID": "template_id",
    "Role framing": "Role_framing",
})

# Keep problem metadata compact. problems.csv has one row per problem/category/template in this dataset,
# so this table deduplicates problem-level columns before joining.
problem_meta = (
    problems
    .rename(columns={"task_id": "problem_id"})
    .sort_values(["problem_id", "category_id"])
    .drop_duplicates("problem_id")
    [["problem_id", "prompt", "test_cases", "challenge_test_list"]]
)

df = (
    results
    .merge(prompt_matrix, on="template_id", how="left", validate="many_to_one")
    .merge(problem_meta, on="problem_id", how="left", validate="many_to_one", suffixes=("", "_problem"))
)

print(f"problems.csv: {problems.shape}")
print(f"prompt matrix: {prompt_matrix.shape}")
print(f"result files: {len(result_paths)} -> {[p.name for p in result_paths]}")
print(f"joined rows: {df.shape}")

display(df.head(3))

In [ ]:
# Basic join checks.
missing_templates = df["Tone"].isna().sum() if "Tone" in df else len(df)
missing_problems = df["prompt"].isna().sum()
print(f"Rows missing prompt-template metadata: {missing_templates}")
print(f"Rows missing problem metadata: {missing_problems}")
print("Rows by model:")
display(df["model_short"].value_counts().rename_axis("model").to_frame("runs"))
print("Templates per model:")
display(df.groupby("model_short")["template_id"].nunique().to_frame("n_templates"))
print("Problems per model:")
display(df.groupby("model_short")["problem_id"].nunique().to_frame("n_problems"))

## Derived Metrics

Performance metrics:

- `passed_int`: exact pass/fail as 0/1
- `tests_fraction`: fraction of unit tests passed

Behavior metrics:

- generation latency and token counts
- code extraction success and code length
- response verbosity beyond extracted code
- markdown-fence usage
- broad error categories from evaluator messages

In [ ]:
def safe_len(value: object) -> int:
    if isinstance(value, str):
        return len(value)
    if pd.isna(value):
        return 0
    return len(str(value))


def count_lines(value: object) -> int:
    if not isinstance(value, str) or not value.strip():
        return 0
    return len(value.splitlines())


def error_text(errors: object) -> str:
    if isinstance(errors, list):
        return "\n".join(map(str, errors))
    if pd.isna(errors):
        return ""
    return str(errors)


def classify_error(errors: object, extracted_code: object) -> str:
    text = error_text(errors).lower()
    code = extracted_code if isinstance(extracted_code, str) else ""
    if not code.strip():
        return "missing_or_empty_code"
    if not text.strip():
        return "none"
    if "not defined" in text:
        return "name_not_defined"
    if "syntax" in text or "invalid syntax" in text:
        return "syntax_error"
    if "timeout" in text or "timed out" in text:
        return "timeout"
    if "importerror" in text or "no module named" in text:
        return "import_error"
    if "assert" in text or "test failed" in text:
        return "wrong_answer"
    return "other_runtime"


df["passed_int"] = df["passed"].astype(bool).astype(int)
df["tests_passed"] = pd.to_numeric(df["tests_passed"], errors="coerce")
df["tests_total"] = pd.to_numeric(df["tests_total"], errors="coerce")
df["tests_fraction"] = np.where(df["tests_total"] > 0, df["tests_passed"] / df["tests_total"], np.nan)
df["prompt_tokens"] = pd.to_numeric(df["prompt_tokens"], errors="coerce")
df["completion_tokens"] = pd.to_numeric(df["completion_tokens"], errors="coerce")
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df["total_tokens"] = df["prompt_tokens"] + df["completion_tokens"]

df["response_chars"] = df["response_text"].map(safe_len)
df["code_chars"] = df["extracted_code"].map(safe_len)
df["response_lines"] = df["response_text"].map(count_lines)
df["code_lines"] = df["extracted_code"].map(count_lines)
df["has_extracted_code"] = (df["code_chars"] > 0).astype(int)
df["has_markdown_fence"] = df["response_text"].fillna("").str.contains(r"```", regex=True).astype(int)
df["extra_text_chars"] = (df["response_chars"] - df["code_chars"]).clip(lower=0)
df["extra_text_ratio"] = np.where(df["response_chars"] > 0, df["extra_text_chars"] / df["response_chars"], np.nan)
df["likely_code_only"] = ((df["has_extracted_code"] == 1) & (df["extra_text_ratio"] < 0.15)).astype(int)
df["error_text"] = df["errors"].map(error_text)
df["error_category"] = [classify_error(e, c) for e, c in zip(df["errors"], df["extracted_code"])]

technique_candidates = ["Tone", "Pressure", "Role_framing", "Format"]
technique_factors = []
for col in technique_candidates:
    if col in df.columns:
        non_null_unique = df[col].dropna().nunique()
        if non_null_unique > 1:
            technique_factors.append(col)
        else:
            print(f"Skipping {col}: only {non_null_unique} non-null value(s).")
print("Technique factors used:", technique_factors)

metric_cols = [
    "passed_int", "tests_fraction", "duration_seconds", "prompt_tokens", "completion_tokens",
    "total_tokens", "has_extracted_code", "has_markdown_fence", "likely_code_only",
    "extra_text_ratio", "code_lines", "response_lines",
]

display(df[["model_short", "problem_id", "template_id", *technique_factors, *metric_cols, "error_category"]].head())

## Overall Performance by Model

In [ ]:
def mean_ci(series: pd.Series, z: float = 1.96) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").dropna()
    n = len(values)
    mean = values.mean() if n else np.nan
    se = values.std(ddof=1) / math.sqrt(n) if n > 1 else np.nan
    return pd.Series({"mean": mean, "ci_low": mean - z * se if n > 1 else np.nan, "ci_high": mean + z * se if n > 1 else np.nan, "n": n})

model_summary = (
    df.groupby("model_short")
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        avg_tests_fraction=("tests_fraction", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        code_extraction_rate=("has_extracted_code", "mean"),
        code_only_rate=("likely_code_only", "mean"),
    )
    .sort_values("pass_rate", ascending=False)
)
display(model_summary.style.format({
    "pass_rate": "{:.1%}",
    "avg_tests_fraction": "{:.1%}",
    "avg_duration_seconds": "{:.2f}",
    "avg_completion_tokens": "{:.1f}",
    "code_extraction_rate": "{:.1%}",
    "code_only_rate": "{:.1%}",
}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x="model_short", y="passed_int", errorbar=("ci", 95), ax=axes[0])
axes[0].set_title("Pass Rate by Model")
axes[0].set_ylabel("Pass rate")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=df, x="model_short", y="tests_fraction", errorbar=("ci", 95), ax=axes[1])
axes[1].set_title("Average Test Fraction by Model")
axes[1].set_ylabel("Tests passed fraction")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## Main Effects of Prompt Techniques

These tables compare each technique level directly. They are useful for quick inspection, but they do not fully control for model or problem difficulty. The paired-effect section below is better for estimating effect direction.

In [ ]:
def summarize_by_factor(data: pd.DataFrame, factor: str, metrics: Iterable[str]) -> pd.DataFrame:
    parts = []
    for metric in metrics:
        summary = (
            data.groupby(factor, dropna=False)[metric]
            .apply(mean_ci)
            .unstack()
            .reset_index()
        )
        summary.insert(1, "metric", metric)
        parts.append(summary)
    return pd.concat(parts, ignore_index=True)

focus_metrics = ["passed_int", "tests_fraction", "duration_seconds", "completion_tokens", "likely_code_only", "extra_text_ratio"]

for factor in technique_factors:
    print(f"\n=== {factor} ===")
    table = summarize_by_factor(df, factor, focus_metrics)
    display(table.style.format({"mean": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}"}))

In [ ]:
# Visual overview of technique effects on correctness.
if technique_factors:
    fig, axes = plt.subplots(1, len(technique_factors), figsize=(5 * len(technique_factors), 4), squeeze=False)
    for ax, factor in zip(axes[0], technique_factors):
        sns.barplot(data=df, x=factor, y="tests_fraction", hue="model_short", errorbar=("ci", 95), ax=ax)
        ax.set_title(f"Tests Passed Fraction by {factor}")
        ax.set_xlabel("")
        ax.set_ylabel("Tests fraction")
        ax.tick_params(axis="x", rotation=25)
        ax.legend(title="Model", fontsize=8)
    plt.tight_layout()
    plt.show()

## Paired Marginal Effects

For each factor level, this computes the difference from a reference level within each `(model, problem)` pair, then averages those paired differences. This is a practical way to reduce confounding from problem difficulty and model baseline strength.

References default to the first level in the prompt matrix ordering. For this dataset that is usually `neutral`, `no role`, and `plaintext`.

In [ ]:
def ordered_levels(data: pd.DataFrame, factor: str) -> list:
    template_order = (
        data[["template_id", factor]]
        .drop_duplicates()
        .sort_values("template_id")[factor]
        .dropna()
        .tolist()
    )
    seen = []
    for level in template_order:
        if level not in seen:
            seen.append(level)
    return seen


def paired_effects(data: pd.DataFrame, factor: str, metric: str, reference: object | None = None) -> pd.DataFrame:
    levels = ordered_levels(data, factor)
    if not levels:
        return pd.DataFrame()
    if reference is None:
        reference = levels[0]
    grouped = (
        data.groupby(["model_short", "problem_id", factor], dropna=False)[metric]
        .mean()
        .reset_index()
    )
    wide = grouped.pivot_table(index=["model_short", "problem_id"], columns=factor, values=metric, aggfunc="mean")
    if reference not in wide.columns:
        raise ValueError(f"Reference {reference!r} not found for {factor}")
    rows = []
    for level in levels:
        if level == reference or level not in wide.columns:
            continue
        delta = (wide[level] - wide[reference]).dropna()
        n = len(delta)
        mean = delta.mean() if n else np.nan
        se = delta.std(ddof=1) / math.sqrt(n) if n > 1 else np.nan
        rows.append({
            "factor": factor,
            "metric": metric,
            "reference": reference,
            "level": level,
            "mean_delta": mean,
            "ci_low": mean - 1.96 * se if n > 1 else np.nan,
            "ci_high": mean + 1.96 * se if n > 1 else np.nan,
            "n_pairs": n,
        })
    return pd.DataFrame(rows)

paired_tables = []
for factor in technique_factors:
    for metric in focus_metrics:
        paired_tables.append(paired_effects(df, factor, metric))
paired_summary = pd.concat(paired_tables, ignore_index=True) if paired_tables else pd.DataFrame()

# Sort by performance impact first, then behavior metrics.
metric_order = {metric: i for i, metric in enumerate(focus_metrics)}
paired_summary["metric_order"] = paired_summary["metric"].map(metric_order)
paired_summary = paired_summary.sort_values(["metric_order", "factor", "mean_delta"], ascending=[True, True, False]).drop(columns="metric_order")
paired_summary.to_csv(OUTPUT_DIR / "paired_prompt_technique_effects.csv", index=False)

display(paired_summary.style.format({"mean_delta": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}"}))
print(f"Saved: {OUTPUT_DIR / 'paired_prompt_technique_effects.csv'}")

In [ ]:
# Plot paired effects for correctness and behavior.
plot_metrics = ["passed_int", "tests_fraction", "completion_tokens", "likely_code_only"]
plot_data = paired_summary[paired_summary["metric"].isin(plot_metrics)].copy()
if not plot_data.empty:
    g = sns.catplot(
        data=plot_data,
        x="mean_delta",
        y="level",
        col="metric",
        row="factor",
        kind="bar",
        sharex=False,
        sharey=False,
        height=3,
        aspect=1.35,
        color="#4C78A8",
    )
    g.set_axis_labels("Mean paired delta vs reference", "Level")
    for ax in g.axes.flat:
        ax.axvline(0, color="black", linewidth=1)
    plt.tight_layout()
    plt.show()

## Model-Specific Technique Effects

This view asks whether a technique helps one model but hurts another.

In [ ]:
def model_factor_summary(data: pd.DataFrame, factor: str, metric: str = "tests_fraction") -> pd.DataFrame:
    return (
        data.groupby(["model_short", factor], dropna=False)
        .agg(
            runs=(metric, "size"),
            mean=(metric, "mean"),
            pass_rate=("passed_int", "mean"),
            avg_completion_tokens=("completion_tokens", "mean"),
            code_only_rate=("likely_code_only", "mean"),
        )
        .reset_index()
        .sort_values(["model_short", "mean"], ascending=[True, False])
    )

for factor in technique_factors:
    print(f"\n=== Model-specific summary: {factor} ===")
    table = model_factor_summary(df, factor)
    display(table.style.format({
        "mean": "{:.1%}",
        "pass_rate": "{:.1%}",
        "avg_completion_tokens": "{:.1f}",
        "code_only_rate": "{:.1%}",
    }))

## Technique Interactions

The prompt matrix is factorial, so combinations matter. These heatmaps show average test fraction across two factors at a time, optionally faceted by model.

In [ ]:
def interaction_heatmap(data: pd.DataFrame, row_factor: str, col_factor: str, metric: str = "tests_fraction") -> None:
    models = sorted(data["model_short"].unique())
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4), squeeze=False)
    for ax, model in zip(axes[0], models):
        subset = data[data["model_short"] == model]
        pivot = subset.pivot_table(index=row_factor, columns=col_factor, values=metric, aggfunc="mean")
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
        ax.set_title(f"{model}: {metric}")
    plt.tight_layout()
    plt.show()

if {"Tone", "Format"}.issubset(technique_factors):
    interaction_heatmap(df, "Tone", "Format", "tests_fraction")
if {"Role_framing", "Format"}.issubset(technique_factors):
    interaction_heatmap(df, "Role_framing", "Format", "tests_fraction")
if {"Tone", "Role_framing"}.issubset(technique_factors):
    interaction_heatmap(df, "Tone", "Role_framing", "tests_fraction")

## Fixed-Effect Models

These linear fixed-effect models estimate prompt-technique associations while controlling for `model_short` and `problem_id`. Treat them as descriptive, not causal proof. Coefficients are easiest to read as average changes relative to the reference categories.

In [ ]:
def fit_fixed_effect(metric: str, interactions: bool = False):
    if not STATSMODELS_AVAILABLE:
        print("statsmodels is not available in this environment.")
        return None
    factors = [f"C({factor})" for factor in technique_factors]
    if interactions and len(technique_factors) >= 2:
        # Main effects plus pairwise interactions among prompt techniques.
        prompt_terms = " + ".join(factors)
        interaction_terms = " + ".join(
            f"C({a}):C({b})" for i, a in enumerate(technique_factors) for b in technique_factors[i + 1:]
        )
        rhs = f"{prompt_terms} + {interaction_terms} + C(model_short) + C(problem_id)"
    else:
        rhs = " + ".join([*factors, "C(model_short)", "C(problem_id)"])
    formula = f"{metric} ~ {rhs}"
    model_data = df[[metric, "model_short", "problem_id", *technique_factors]].dropna()
    result = smf.ols(formula, data=model_data).fit(cov_type="HC3")
    print(formula)
    return result

regression_metrics = ["passed_int", "tests_fraction", "duration_seconds", "completion_tokens", "likely_code_only", "extra_text_ratio"]
regression_rows = []
for metric in regression_metrics:
    result = fit_fixed_effect(metric)
    if result is None:
        continue
    params = result.params.rename("coef").to_frame()
    params["std_err"] = result.bse
    params["p_value"] = result.pvalues
    params["metric"] = metric
    params["term"] = params.index
    regression_rows.append(params.reset_index(drop=True))

regression_table = pd.concat(regression_rows, ignore_index=True) if regression_rows else pd.DataFrame()
technique_terms = regression_table[regression_table["term"].str.contains("Tone|Role_framing|Format|Pressure", regex=True, na=False)].copy()
technique_terms.to_csv(OUTPUT_DIR / "fixed_effect_prompt_technique_models.csv", index=False)

display(technique_terms.style.format({"coef": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}"}))
print(f"Saved: {OUTPUT_DIR / 'fixed_effect_prompt_technique_models.csv'}")

## Error Modes

This section checks whether prompt techniques change the kind of failures models produce, not just whether they pass.

In [ ]:
error_overview = (
    df.groupby(["model_short", "error_category"])
    .size()
    .rename("runs")
    .reset_index()
)
error_overview["share_within_model"] = error_overview["runs"] / error_overview.groupby("model_short")["runs"].transform("sum")
display(error_overview.sort_values(["model_short", "runs"], ascending=[True, False]).style.format({"share_within_model": "{:.1%}"}))

for factor in technique_factors:
    print(f"\n=== Error category by {factor} ===")
    table = pd.crosstab(df[factor], df["error_category"], normalize="index")
    display(table.style.format("{:.1%}"))

fig, ax = plt.subplots(figsize=(10, 4))
sns.countplot(data=df, x="error_category", hue="model_short", ax=ax)
ax.set_title("Error Categories by Model")
ax.set_xlabel("")
ax.set_ylabel("Runs")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## Best and Worst Prompt Templates

This ranks complete prompt templates, which helps identify whether the winning behavior comes from a single factor or a combination.

In [ ]:
template_summary = (
    df.groupby(["template_id", *technique_factors])
    .agg(
        runs=("task_id", "size"),
        models=("model_short", "nunique"),
        problems=("problem_id", "nunique"),
        pass_rate=("passed_int", "mean"),
        tests_fraction=("tests_fraction", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        code_extraction_rate=("has_extracted_code", "mean"),
    )
    .reset_index()
    .sort_values(["tests_fraction", "pass_rate"], ascending=False)
)
template_summary.to_csv(OUTPUT_DIR / "prompt_template_summary.csv", index=False)

display(template_summary.head(10).style.format({
    "pass_rate": "{:.1%}",
    "tests_fraction": "{:.1%}",
    "avg_duration_seconds": "{:.2f}",
    "avg_completion_tokens": "{:.1f}",
    "code_only_rate": "{:.1%}",
    "code_extraction_rate": "{:.1%}",
}))
print("Worst templates:")
display(template_summary.tail(10).style.format({
    "pass_rate": "{:.1%}",
    "tests_fraction": "{:.1%}",
    "avg_duration_seconds": "{:.2f}",
    "avg_completion_tokens": "{:.1f}",
    "code_only_rate": "{:.1%}",
    "code_extraction_rate": "{:.1%}",
}))
print(f"Saved: {OUTPUT_DIR / 'prompt_template_summary.csv'}")

## Inspect Concrete Examples

Use this cell to examine representative successes/failures for a model, template, or technique level.

In [ ]:
def show_examples(
    data: pd.DataFrame,
    *,
    model: str | None = None,
    factor: str | None = None,
    level: str | None = None,
    passed: bool | None = None,
    n: int = 3,
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    subset = data.copy()
    if model is not None:
        subset = subset[subset["model_short"] == model]
    if factor is not None and level is not None:
        subset = subset[subset[factor] == level]
    if passed is not None:
        subset = subset[subset["passed_int"] == int(passed)]
    cols = [
        "model_short", "problem_id", "template_id", *technique_factors,
        "passed_int", "tests_passed", "tests_total", "error_category",
        "response_text", "extracted_code",
    ]
    if len(subset) == 0:
        return subset[cols]
    return subset.sample(min(n, len(subset)), random_state=random_state)[cols]

# Example usage:
# display(show_examples(df, model="gemma2-9b", factor="Format", level="json", passed=False, n=2))
display(show_examples(df, n=2))

## Auto-Written Interpretation Aids

These tables surface likely takeaways. Read them together with the confidence intervals and model-specific plots above.

In [ ]:
def top_effects(summary: pd.DataFrame, metric: str, k: int = 8) -> pd.DataFrame:
    subset = summary[summary["metric"] == metric].copy()
    if subset.empty:
        return subset
    subset["abs_delta"] = subset["mean_delta"].abs()
    return subset.sort_values("abs_delta", ascending=False).head(k).drop(columns="abs_delta")

print("Largest paired effects on tests_fraction:")
display(top_effects(paired_summary, "tests_fraction").style.format({"mean_delta": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}"}))

print("Largest paired effects on pass/fail:")
display(top_effects(paired_summary, "passed_int").style.format({"mean_delta": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}"}))

print("Largest paired behavior effects on completion_tokens:")
display(top_effects(paired_summary, "completion_tokens").style.format({"mean_delta": "{:+.1f}", "ci_low": "{:+.1f}", "ci_high": "{:+.1f}"}))

print("Largest paired behavior effects on likely_code_only:")
display(top_effects(paired_summary, "likely_code_only").style.format({"mean_delta": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}"}))

## Notes for Reporting

When writing up findings, prefer this structure:

1. Start with the paired marginal effects for `tests_fraction` and `passed_int`.
2. Check whether the effect is consistent across models in the model-specific summaries.
3. Use behavior metrics to explain *how* the prompt changed outputs: verbosity, code-only behavior, extraction failures, token use, latency, and error categories.
4. Treat fixed-effect regression coefficients as supporting evidence, especially when signs match the paired effects.
5. Avoid over-interpreting `Pressure` here if it is skipped as constant or empty in the current prompt matrix.